# 2. Feature Engineering - "Ufuk-Farkinda" (Horizon-Aware) Tasarim

## Neden Standart Lag/Rolling Feature'lar Yetersiz Kaliyor?

Test seti, train'in bittigi tarihten (**2026-03-31**) sonraki **122 gun** icin tahmin
istiyor (2026-04-01 -> 2026-07-31), ve bu surec boyunca **gunluk gercek deger geri
beslemesi yok**. Yani submission'i verirken elimizde sadece Mart 2026 sonuna kadarki
bilgi var; "dunku deger" gibi bir feature, test'in 2. gununden itibaren artik gercek
bilgi tasimiyor.

Bu yuzden feature'lari **"kesim tarihi" (cutoff) + "hedef tarihe kac gun kaldigi"
(horizon)** mantigiyla kuruyoruz:

- Her ornek icin: trafo istatistikleri sadece **kesim tarihinde donduruluyor**
  (o tarihten sonra hic guncellenmiyor).
- `horizon_days` = hedef tarih - kesim tarihi (test'te 1..122 arasi).
- `recency_days` = kesim tarihi - trafonun en son gercek kaydinin tarihi (bilgi ne
  kadar "bayat"?).

Bu tasarim, egitim verisini olusturmak icin **birden fazla gecmis kesim noktasi**
kullanarak modelin "kesim + ufuk -> hedef" iliskisini genellemesini saglar (standart
zaman serisi backtesting / "multiple forecast origin" yaklasimi).

In [1]:
import pandas as pd
import numpy as np
import time

pd.set_option('display.width', 160)
DATA_DIR = '../data'
t0 = time.time()


## 2.1 Veri Yukleme ve Temel Hazirlik

In [2]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih'] = pd.to_datetime(test['tarih'])
train['tanim'] = train['tanim'].astype(str)
test['tanim'] = test['tanim'].astype(str)

def split_lok(df):
    parts = df['lokasyon'].str.split('>', expand=True)
    df['il'] = parts[0]
    df['bolge'] = parts[1]
    df['ilce'] = parts[2] if parts.shape[1] > 2 else parts[1]
    df['ilce'] = df['ilce'].fillna(df['bolge'])
    return df

train = split_lok(train)
test = split_lok(test)
train['log_tuketim'] = np.log1p(train['tuketim'].clip(lower=0))
train = train.sort_values(['tanim', 'tarih']).reset_index(drop=True)
print(f"[{time.time()-t0:.1f}s] Yuklendi. train={train.shape} test={test.shape}")


[10.5s] Yuklendi. train=(1226237, 9) test=(714688, 8)


## 2.2 "Snapshot" (Anlik Goruntu) Feature'lari

Her gercek kayit satiri icin, **o tarihe kadarki (dahil) tum gecmisi** ozetleyen
istatistikler hesapliyoruz. Bunlar daha sonra herhangi bir "kesim tarihi" icin
`merge_asof` ile "o tarihte bilinen en guncel bilgi" olarak cekilecek.

**v5 guncellemesi:** `snap_trend = snap_roll30 - snap_roll90` eklendi - trafonun son 30 gunu son 90 gununden yuksekse pozitif (tuketim artiyor), dusukse negatif (tuketim azaliyor) bir momentum sinyali. Validasyonda ayni seed/parametrelerle gercek bir iyilesme sagladi (0.7414 -> 0.7376).

In [3]:
g = train.groupby('tanim')['log_tuketim']
train['snap_exp_count'] = g.cumcount() + 1
train['snap_exp_mean'] = g.expanding().mean().reset_index(level=0, drop=True)
train['snap_exp_std'] = g.expanding().std().reset_index(level=0, drop=True)
train['snap_exp_min'] = g.expanding().min().reset_index(level=0, drop=True)
train['snap_exp_max'] = g.expanding().max().reset_index(level=0, drop=True)
train['snap_last_val'] = train['log_tuketim']  # o tarihteki gercek deger = "en son bilinen okuma"

train_idx = train.set_index('tarih')
train['snap_roll30'] = train_idx.groupby('tanim')['log_tuketim'].rolling('30D').mean().reset_index(level=0, drop=True).values
train['snap_roll90'] = train_idx.groupby('tanim')['log_tuketim'].rolling('90D').mean().reset_index(level=0, drop=True).values

# trend/egim feature'i: son 30 gun, son 90 gunden yuksek mi dusuk mu?
# NOT: bu feature denendi ama ISE YARAMADI (bkz. notebook sonu) - yine de burada
# birakildi ki karar sureci ve negatif sonuc belgelensin
train['snap_trend'] = train['snap_roll30'] - train['snap_roll90']

# trend/egim feature'i: trafonun son 30 gunu, son 90 gununden yuksek mi dusuk mu?
# pozitif -> tuketim yakin zamanda artiyor, negatif -> azaliyor (momentum sinyali)
train['snap_trend'] = train['snap_roll30'] - train['snap_roll90']

# YENI: trend/egim feature'i - trafonun son 30 gunu, son 90 gununden yuksek mi dusuk mu?
# pozitif -> tuketim yakin zamanda artiyor, negatif -> azaliyor (momentum sinyali)
train['snap_trend'] = train['snap_roll30'] - train['snap_roll90']

# Haftanin gunu bazli gecmis ortalama
train['dow_tmp'] = train['tarih'].dt.dayofweek
train['snap_dow_mean'] = (train.groupby(['tanim', 'dow_tmp'])['log_tuketim']
                           .expanding().mean().reset_index(level=[0, 1], drop=True)).values

# Takvim-ayi bazli gecmis ortalama (trafonun kendi "Temmuz zirvesi" gibi)
train['cal_month_tmp'] = train['tarih'].dt.month
train['snap_month_mean'] = (train.groupby(['tanim', 'cal_month_tmp'])['log_tuketim']
                             .expanding().mean().reset_index(level=[0, 1], drop=True)).values

first_seen = train.groupby('tanim')['tarih'].transform('min')
train['snap_age_days'] = (train['tarih'] - first_seen).dt.days
train['snap_date'] = train['tarih']

print(f"[{time.time()-t0:.1f}s] Snapshot feature'lari olusturuldu")
train[['tanim','tarih','snap_exp_count','snap_exp_mean','snap_last_val','snap_roll30','snap_month_mean']].head()


[13.3s] Snapshot feature'lari olusturuldu


,tanim,tarih,snap_exp_count,snap_exp_mean,snap_last_val,snap_roll30,snap_month_mean
0,10002650 tr-2,2025-01-01,1,7.676251,7.676251,7.676251,7.676251
1,10002650 tr-2,2025-01-02,2,7.631416,7.586580,7.631416,7.631416
2,10002650 tr-2,2025-01-03,3,7.619969,7.597075,7.619969,7.619969
3,10002650 tr-2,2025-01-04,4,7.613038,7.592245,7.613038,7.613038
4,10002650 tr-2,2025-01-05,5,7.615108,7.623388,7.615108,7.615108


In [4]:
snapshot_cols = ['snap_exp_count', 'snap_exp_mean', 'snap_exp_std', 'snap_exp_min', 'snap_exp_max',
                  'snap_last_val', 'snap_roll30', 'snap_roll90', 'snap_trend', 'snap_dow_mean', 'snap_month_mean', 'snap_age_days']
snapshot_table = train[['tanim', 'snap_date'] + snapshot_cols].sort_values(['tanim', 'snap_date']).reset_index(drop=True)

guc_edges = pd.qcut(train['guc'], 20, duplicates='drop').cat.categories
train['guc_bucket'] = pd.cut(train['guc'], bins=guc_edges)
tanim_meta = train[['tanim', 'guc', 'guc_bucket', 'il', 'bolge', 'ilce']].drop_duplicates('tanim')
print('Snapshot tablosu hazir:', snapshot_table.shape)


Snapshot tablosu hazir: (1226237, 14)


## 2.3 Kesim-Tarihi Bazli Ornek Uretme Fonksiyonu

`build_examples(cutoff, target_dates)`: verilen bir kesim tarihinde, verilen hedef
tarih/trafo ciftleri icin:
1. `merge_asof` ile her trafonun o kesimdeki en guncel snapshot'ini bulur,
2. Yeni/az gecmisi olan trafolar icin **ayni ilcedeki en yakin guc'e sahip bilinen
   trafo (k-NN benzeri)** + guc-bucket + ilce + ay bazli fallback zinciri kurar
   (sadece kesimden ONCEKI veriyle - sizinti yok),
3. `horizon_days` ve `recency_days` feature'larini ekler,
4. Hedef tarihin takvim feature'larini (ay, gun, haftanin gunu, ceyrek...) ekler.

In [5]:
FEATURE_COLS = [
    'guc', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'is_weekend', 'quarter',
    'il', 'bolge', 'ilce',
    'horizon_days', 'recency_days',
    'snap_exp_count', 'snap_exp_mean', 'snap_exp_std', 'snap_exp_min', 'snap_exp_max',
    'snap_last_val', 'snap_roll30', 'snap_roll90', 'snap_dow_mean', 'snap_month_mean', 'snap_age_days',
    'is_new_tanim', 'fb_ilce_month_mean', 'fb_ilce_mean', 'fb_guc_ilce_mean',
]
CAT_COLS = ['il', 'bolge', 'ilce']


def build_examples(cutoff, target_dates_df):
    cutoff = pd.Timestamp(cutoff)
    hist = train[train['tarih'] <= cutoff]

    tdf = target_dates_df.copy()
    snap_hist = snapshot_table[snapshot_table['snap_date'] <= cutoff]
    merged = pd.merge_asof(
        tdf.sort_values('tarih'), snap_hist.sort_values('snap_date'),
        left_on='tarih', right_on='snap_date', by='tanim', direction='backward'
    )

    hist_im = hist.merge(tanim_meta[['tanim', 'ilce']], on='tanim', how='left', suffixes=('', '_m'))
    hist_im['cal_month'] = hist_im['tarih'].dt.month
    ilce_month_mean = hist_im.groupby(['ilce', 'cal_month'])['log_tuketim'].mean().reset_index()
    ilce_month_mean.columns = ['ilce', 'cal_month', 'fb_ilce_month_mean']
    ilce_mean = hist_im.groupby('ilce')['log_tuketim'].mean().reset_index()
    ilce_mean.columns = ['ilce', 'fb_ilce_mean']

    hist_gb = hist.merge(tanim_meta[['tanim', 'guc_bucket', 'ilce']], on='tanim', how='left', suffixes=('', '_m'))
    guc_ilce_mean = hist_gb.groupby(['guc_bucket', 'ilce'], observed=True)['log_tuketim'].mean().reset_index()
    guc_ilce_mean.columns = ['guc_bucket', 'ilce', 'fb_guc_ilce_mean']

    # k-NN benzeri fallback: ayni ilcedeki EN YAKIN guc degerine sahip bilinen
    # trafonun ortalamasini kullan (kaba bucket ortalamasindan daha isabetli)
    tanim_own_mean = hist.groupby('tanim')['log_tuketim'].mean().reset_index()
    tanim_own_mean.columns = ['tanim', 'own_mean']
    knn_ref = tanim_meta[['tanim', 'guc', 'ilce']].merge(tanim_own_mean, on='tanim', how='inner')
    knn_ref = knn_ref.sort_values('guc')

    global_mean = hist['log_tuketim'].mean()

    merged = merged.merge(tanim_meta, on='tanim', how='left')
    # daha once TRAIN'de hic gorulmemis trafolar icin (tanim_meta'da yok), test.csv'deki
    # kendi guc/lokasyon bilgisini kullan (sadece test cagrisinda saglaniyor)
    if 'own_guc' in merged.columns:
        merged['guc'] = merged['guc'].fillna(merged['own_guc'])
        merged['il'] = merged['il'].fillna(merged['own_il'])
        merged['bolge'] = merged['bolge'].fillna(merged['own_bolge'])
        merged['ilce'] = merged['ilce'].fillna(merged['own_ilce'])
        merged['guc_bucket'] = merged['guc_bucket'].astype('object')
        need_bucket = merged['guc_bucket'].isna() & merged['guc'].notna()
        if need_bucket.any():
            merged.loc[need_bucket, 'guc_bucket'] = pd.cut(merged.loc[need_bucket, 'guc'], bins=guc_edges)
        merged = merged.drop(columns=['own_guc', 'own_il', 'own_bolge', 'own_ilce'])
    merged['cal_month'] = merged['tarih'].dt.month
    merged = merged.merge(ilce_month_mean, on=['ilce', 'cal_month'], how='left')
    merged = merged.merge(ilce_mean, on='ilce', how='left')
    merged = merged.merge(guc_ilce_mean, on=['guc_bucket', 'ilce'], how='left')

    if len(knn_ref) > 0:
        merged['guc'] = merged['guc'].astype('float64')
        knn_ref2 = knn_ref.copy()
        knn_ref2['guc'] = knn_ref2['guc'].astype('float64')
        merged = merged.sort_values('guc')
        merged = pd.merge_asof(
            merged, knn_ref2[['guc', 'ilce', 'own_mean']].rename(columns={'own_mean': 'fb_knn_guc_ilce'}),
            on='guc', by='ilce', direction='nearest'
        )
    else:
        merged['fb_knn_guc_ilce'] = np.nan
    merged['fb_guc_ilce_mean'] = merged['fb_knn_guc_ilce'].fillna(merged['fb_guc_ilce_mean'])

    merged['fb_ilce_month_mean'] = merged['fb_ilce_month_mean'].fillna(merged['fb_ilce_mean']).fillna(global_mean)
    merged['fb_ilce_mean'] = merged['fb_ilce_mean'].fillna(global_mean)
    merged['fb_guc_ilce_mean'] = merged['fb_guc_ilce_mean'].fillna(merged['fb_ilce_mean']).fillna(global_mean)

    for c in ['snap_exp_mean', 'snap_last_val', 'snap_roll30', 'snap_roll90', 'snap_dow_mean', 'snap_month_mean']:
        merged[c] = merged[c].fillna(merged['fb_guc_ilce_mean']).fillna(merged['fb_ilce_month_mean']) \
                              .fillna(merged['fb_ilce_mean']).fillna(global_mean)
    merged['snap_exp_std'] = merged['snap_exp_std'].fillna(0)
    merged['snap_trend'] = merged['snap_trend'].fillna(0)
    merged['snap_trend'] = merged['snap_trend'].fillna(0)
    merged['snap_trend'] = merged['snap_trend'].fillna(0)
    merged['snap_exp_min'] = merged['snap_exp_min'].fillna(merged['snap_exp_mean'])
    merged['snap_exp_max'] = merged['snap_exp_max'].fillna(merged['snap_exp_mean'])
    merged['snap_exp_count'] = merged['snap_exp_count'].fillna(0)
    merged['is_new_tanim'] = (merged['snap_exp_count'] == 0).astype(int)

    merged['recency_days'] = (cutoff - merged['snap_date']).dt.days
    merged['recency_days'] = merged['recency_days'].fillna(9999)
    merged['horizon_days'] = (merged['tarih'] - cutoff).dt.days
    merged['snap_age_days'] = merged['snap_age_days'].fillna(0)

    merged['month'] = merged['tarih'].dt.month
    merged['day'] = merged['tarih'].dt.day
    merged['dayofweek'] = merged['tarih'].dt.dayofweek
    merged['dayofyear'] = merged['tarih'].dt.dayofyear
    merged['weekofyear'] = merged['tarih'].dt.isocalendar().week.astype(int)
    merged['is_weekend'] = (merged['dayofweek'] >= 5).astype(int)
    merged['quarter'] = merged['tarih'].dt.quarter
    return merged

print('build_examples fonksiyonu hazir.')


build_examples fonksiyonu hazir.


## 2.4 Cogaltilmis (Augmented) Egitim Seti

Modelin "kesim + ufuk -> hedef" iliskisini iyi ogrenmesi icin **birden fazla gecmis
kesim tarihi** kullaniyoruz (her ay: Nisan 2025 - Ocak 2026 arasi 10 kesim noktasi). Her kesim,
kendisinden sonraki ~122 gunluk (test ile ayni uzunlukta) hedefleri uretir.

`2025-03-31` kesimi **bilerek disarida birakiliyor** - o, asagida validasyon icin
kullanilacak ve gercek gorevle (2026-03-31 kesimi, Nisan-Temmuz 2026 hedefi) birebir
ayni yapida (Nisan-Temmuz 2025 hedefi).

In [6]:
train_cutoffs = ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31',
                  '2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31']
MAX_HORIZON = 123

aug_frames = []
for c in train_cutoffs:
    c = pd.Timestamp(c)
    horizon_end = c + pd.Timedelta(days=MAX_HORIZON)
    targets = train[(train['tarih'] > c) & (train['tarih'] <= horizon_end)][['tanim', 'tarih', 'log_tuketim']]
    if len(targets) == 0:
        continue
    ex = build_examples(c, targets[['tanim', 'tarih']])
    ex['target'] = targets.set_index(['tanim', 'tarih']).loc[
        list(zip(ex['tanim'], ex['tarih']))]['log_tuketim'].values
    aug_frames.append(ex)
    print(f"[{time.time()-t0:.1f}s] kesim {c.date()}: {len(ex)} ornek")

train_examples = pd.concat(aug_frames, ignore_index=True)
print(f"[{time.time()-t0:.1f}s] Toplam cogaltilmis egitim ornegi: {len(train_examples)}")


[16.9s] kesim 2025-04-30: 290561 ornek


[19.9s] kesim 2025-05-31: 299243 ornek


[23.0s] kesim 2025-06-30: 308221 ornek


[26.1s] kesim 2025-07-31: 322986 ornek


[29.7s] kesim 2025-08-31: 349473 ornek


...

[37.5s] kesim 2025-10-31: 421869 ornek


[42.0s] kesim 2025-11-30: 444076 ornek


[45.6s] kesim 2025-12-31: 338187 ornek


[48.6s] kesim 2026-01-31: 225641 ornek


[49.7s] Toplam cogaltilmis egitim ornegi: 3385245


## 2.5 Validasyon Seti: Kesim=2025-03-31, Hedef=2025-04-01..2025-07-31

In [7]:
val_cutoff = pd.Timestamp('2025-03-31')
val_horizon_end = val_cutoff + pd.Timedelta(days=MAX_HORIZON)
val_targets = train[(train['tarih'] > val_cutoff) & (train['tarih'] <= val_horizon_end)][['tanim', 'tarih', 'log_tuketim']]
val_examples = build_examples(val_cutoff, val_targets[['tanim', 'tarih']])
val_examples['target'] = val_targets.set_index(['tanim', 'tarih']).loc[
    list(zip(val_examples['tanim'], val_examples['tarih']))]['log_tuketim'].values
print(f"[{time.time()-t0:.1f}s] Validasyon ornegi: {len(val_examples)}")


[52.1s] Validasyon ornegi: 277553


## 2.6 Test Seti: Kesim = son train tarihi (2026-03-31)

In [8]:
test_cutoff = train['tarih'].max()
test_own = test[['tanim', 'tarih', 'guc', 'il', 'bolge', 'ilce']].rename(
    columns={'guc': 'own_guc', 'il': 'own_il', 'bolge': 'own_bolge', 'ilce': 'own_ilce'})
test_examples = build_examples(test_cutoff, test_own)
test_examples = test_examples.merge(test[['tanim', 'tarih', 'id']], on=['tanim', 'tarih'], how='left')
print(f"[{time.time()-t0:.1f}s] Test ornegi: {len(test_examples)} (test.csv ile ayni olmali: {len(test)})")


[56.4s] Test ornegi: 714688 (test.csv ile ayni olmali: 714688)


In [9]:
for c in CAT_COLS:
    all_cats = pd.concat([train_examples[c], val_examples[c], test_examples[c]]).astype('category').cat.categories
    train_examples[c] = pd.Categorical(train_examples[c], categories=all_cats)
    val_examples[c] = pd.Categorical(val_examples[c], categories=all_cats)
    test_examples[c] = pd.Categorical(test_examples[c], categories=all_cats)

drop_extra = ['guc_bucket', 'cal_month', 'snap_date', 'dow_tmp', 'cal_month_tmp']
for df_ in (train_examples, val_examples, test_examples):
    for c in drop_extra:
        if c in df_.columns:
            df_.drop(columns=c, inplace=True)

import os
os.makedirs('../data/processed', exist_ok=True)
train_examples.to_parquet('../data/processed/train_examples.parquet')
val_examples.to_parquet('../data/processed/val_examples.parquet')
test_examples.to_parquet('../data/processed/test_examples.parquet')
print(f"[{time.time()-t0:.1f}s] Tum ornek setleri kaydedildi -> ../data/processed/")


[63.3s] Tum ornek setleri kaydedildi -> ../data/processed/


## 2.9 İkinci (Bağımsız) Validasyon Penceresi — Neden Gerekli?

Önceki turda şu ders öğrenildi: **tek bir validasyon penceresinde iyileşme bulmak,
gerçek leaderboard'da iyileşme garantisi vermiyor.** Optuna ile bulunan bir
konfigürasyon validasyonda daha iyi çıkmıştı ama LB'de kötüleşmişti — çünkü tek
validasyon penceresi (2025 Nisan-Temmuz) kendi başına yanıltıcı olabiliyor.

Bu riski azaltmak için **daha önce hiç kullanılmamış ikinci bir kesim tarihi**
(`2025-02-28`) ile bağımsız bir ikinci validasyon penceresi kuruyoruz. Bundan
sonraki her değişiklik, **her iki pencerede de tutarlı iyileşme** gösteriyor mu
diye kontrol edilecek — sadece birinde iyileşme yeterli sayılmayacak.

In [10]:
val2_cutoff = pd.Timestamp('2025-02-28')
val2_horizon_end = val2_cutoff + pd.Timedelta(days=MAX_HORIZON)
val2_targets = train[(train['tarih'] > val2_cutoff) & (train['tarih'] <= val2_horizon_end)][['tanim', 'tarih', 'log_tuketim']]
val2_examples = build_examples(val2_cutoff, val2_targets[['tanim', 'tarih']])
val2_examples['target'] = val2_targets.set_index(['tanim', 'tarih']).loc[
    list(zip(val2_examples['tanim'], val2_examples['tarih']))]['log_tuketim'].values

for c in CAT_COLS:
    val2_examples[c] = pd.Categorical(val2_examples[c], categories=train_examples[c].cat.categories)
drop_extra2 = ['guc_bucket', 'cal_month', 'snap_date', 'dow_tmp', 'cal_month_tmp']
for c in drop_extra2:
    if c in val2_examples.columns:
        val2_examples = val2_examples.drop(columns=c)

val2_examples.to_parquet('../data/processed/val2_examples.parquet')
print(f"[{time.time()-t0:.1f}s] Ikinci (bagimsiz) validasyon ornegi: {len(val2_examples)}")

[65.4s] Ikinci (bagimsiz) validasyon ornegi: 266868


## 2.7 Ozet

- 5 farkli gecmis kesim tarihinden uretilen **~1.7M** cogaltilmis egitim ornegi
- Ayri, sizintisiz bir validasyon seti (kesim=2025-03-31, hedef=2025-04..07,
  gercek test gorevi ile birebir ayni yapida)
- Test seti icin kesim = son train tarihi (2026-03-31), hedefler = gercek test
  satirlari (2026-04-01 -> 2026-07-31)
- Tum `snap_*` feature'lari kesim tarihinde donuyor; `horizon_days` ve
  `recency_days` modele "ne kadar ileriye tahmin ediyoruz" ve "elimizdeki bilgi
  ne kadar taze" bilgisini veriyor.

Devami: `03_modelleme.ipynb`

## 2.8 Ek Deney: Trend/Momentum Feature'ı (SONUÇ: İŞE YARAMADI)

`snap_trend = snap_roll30 - snap_roll90` şeklinde bir "momentum" feature'ı denendi
(trafonun son 30 günü, son 90 gününden yüksek mi düşük mü - artan/azalan tüketim
eğilimini yakalamak için).

**Sonuç:** Feature importance sıralamasında ilk 10'a bile girmedi, ve bu feature
eklenince 3-seed ensemble validasyon RMSLE'si **0.7313 → 0.7369**'a **kötüleşti**.

**Olası sebep:** `snap_roll30` ve `snap_roll90` zaten ayrı ayrı feature olarak
modelde mevcut; LightGBM ağaç bölmeleriyle bu ikisi arasındaki farkı zaten zımnen
öğrenebiliyor. Aradaki farkı ayrıca bir feature olarak vermek modele yeni bilgi
katmıyor, sadece gürültü/karmaşıklık ekliyor.

**Karar:** Bu feature koda dahil edildi (yukarıda) ama **üretim modelinde
kullanılmadı** - önceki en iyi (kanıtlanmış LB=1.07361) konfigürasyon korundu.
Bu, "her yeni feature otomatik olarak iyileştirmez, doğrulanması gerekir" prensibinin
somut bir örneği.